In [ ]:
!unzip -o /content/drive/MyDrive/beacon_yelp/restaurant_data_package.zip -d /content/
!ls /content/dataset
!pip install transformers datasets scikit-learn -q

Archive:  /content/drive/MyDrive/beacon_yelp/restaurant_data_package.zip
  inflating: /content/dataset/restaurant_test_30k.json  
  inflating: /content/dataset/restaurant_train_70k.json  
  inflating: /content/dataset/synthetic_reviews_multi_style_clipped.csv  
  inflating: /content/dataset/embeddings_base_70k.npy  
  inflating: /content/dataset/embeddings_synthetic_multi_style_clipped.npy  
embeddings_base_70k.npy
embeddings_synthetic_multi_style_clipped.npy
restaurant_test_30k.json
restaurant_train_70k.json
synthetic_reviews_multi_style_clipped.csv


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict

GUIDE_SIZE             = 3000
SYNTH_SIZE             = 30000
RANDOM_STATE           = 42
SAMPLES_PER_STAR_GUIDE = GUIDE_SIZE  // 5
SAMPLES_PER_STAR_SYNTH = SYNTH_SIZE  // 5

# ts — synthetic
synth_df   = pd.read_csv('/content/dataset/synthetic_reviews_multi_style_clipped.csv')
synth_df   = synth_df.rename(columns={'full_text': 'text', 'stars': 'label'})
synth_df['label'] = synth_df['label'].astype(float)
synth_embs = np.load('/content/dataset/embeddings_synthetic_multi_style_clipped.npy')
assert len(synth_embs) == len(synth_df)
synth_df['emb_idx'] = np.arange(len(synth_df))

synth_sampled = (
    synth_df.groupby('label', group_keys=False)
            .apply(lambda x: x.sample(n=min(SAMPLES_PER_STAR_SYNTH, len(x)), random_state=RANDOM_STATE))
            .reset_index(drop=True)
)
synth_embs_sampled = synth_embs[synth_sampled['emb_idx'].values]
synth_sampled      = synth_sampled.drop(columns=['emb_idx'])
print(f"ts distribution:\n{synth_sampled['label'].value_counts().sort_index().to_string()}")

# tg_guide — real train
guide_full_df   = pd.read_json('/content/dataset/restaurant_train_70k.json', lines=True)
guide_full_df   = guide_full_df.rename(columns={'stars': 'label'})
guide_full_df['label'] = guide_full_df['label'].astype(float)
guide_full_embs = np.load('/content/dataset/embeddings_base_70k.npy')
assert len(guide_full_embs) == len(guide_full_df)
guide_full_df['emb_idx'] = np.arange(len(guide_full_df))

guide_df = (
    guide_full_df.groupby('label', group_keys=False)
                 .apply(lambda x: x.sample(n=min(SAMPLES_PER_STAR_GUIDE, len(x)), random_state=RANDOM_STATE))
                 .reset_index(drop=True)
)
guide_embs = guide_full_embs[guide_df['emb_idx'].values]
guide_df   = guide_df.drop(columns=['emb_idx'])
print(f"\ntg_guide distribution:\n{guide_df['label'].value_counts().sort_index().to_string()}")

# tg_eval — locked test set
eval_df = pd.read_json('/content/dataset/restaurant_test_30k.json', lines=True)
eval_df = eval_df.rename(columns={'stars': 'label'})
eval_df['label'] = eval_df['label'].astype(float)
print(f"\ntg_eval distribution:\n{eval_df['label'].value_counts().sort_index().to_string()}")

# ts train/val split
train_df   = synth_sampled.sample(frac=0.7, random_state=RANDOM_STATE)
val_df     = synth_sampled.drop(train_df.index)
train_embs = synth_embs_sampled[train_df.index.values]
val_embs   = synth_embs_sampled[val_df.index.values]
train_df   = train_df.reset_index(drop=True)
val_df     = val_df.reset_index(drop=True)

ds = DatasetDict({
    'train':      Dataset.from_pandas(train_df[['text', 'label']]),
    'validation': Dataset.from_pandas(val_df[['text', 'label']]),
    'guide':      Dataset.from_pandas(guide_df[['text', 'label']]),
    'test':       Dataset.from_pandas(eval_df[['text', 'label']].reset_index(drop=True))
})

print(f"\n{'Split':<14}{'Size':<10}")
print("-" * 24)
for split in ['train', 'validation', 'guide', 'test']:
    print(f"{split:<14}{len(ds[split]):<10}")
print(f"\ntrain_embs: {train_embs.shape} | guide_embs: {guide_embs.shape}")

/tmp/ipykernel_2543/1028458001.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(SAMPLES_PER_STAR_SYNTH, len(x)), random_state=RANDOM_STATE))


ts distribution:
label
1.0    5950
2.0    5949
3.0    5980
4.0    5960
5.0    5952


/tmp/ipykernel_2543/1028458001.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(SAMPLES_PER_STAR_GUIDE, len(x)), random_state=RANDOM_STATE))



tg_guide distribution:
label
1.0    600
2.0    600
3.0    600
4.0    600
5.0    600

tg_eval distribution:
label
1.0    6000
2.0    6000
3.0    6000
4.0    6000
5.0    6000

Split         Size      
------------------------
train         20854     
validation    8937      
guide         3000      
test          30000     

train_embs: (20854, 384) | guide_embs: (3000, 384)


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
import os, json

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LEN    = 256
BATCH_SIZE = 64
LR         = 2e-5

# K ablation config
K_VALUES     = [1, 10, 50, 100, 'all']
NUM_EPOCHS   = 10
PATIENCE     = 2
BASE_DIR     = "/content/k_ablation"
BASELINE_QWK = 0.8051
os.makedirs(BASE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    cap   = torch.cuda.get_device_capability()
    DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
else:
    DTYPE = torch.float32

print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Dtype:  {DTYPE}")
print(f"K values to run: {K_VALUES}")

Device: cuda | GPU: NVIDIA A100-SXM4-40GB
Dtype:  torch.bfloat16
K values to run: [1, 10, 50, 100, 'all']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

ds_tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
print("Tokenization done.")
print(ds_tokenized)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/20854 [00:00<?, ? examples/s]

Map:   0%|          | 0/8937 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Tokenization done.
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20854
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8937
    })
    guide: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30000
    })
})


In [ ]:
def compute_qwk(preds, labels):
    preds_rounded = np.clip(np.round(preds), 1, 5).astype(int)
    labels_int    = np.array(labels).astype(int)
    return cohen_kappa_score(labels_int, preds_rounded, weights="quadratic")

print("QWK sanity check:", compute_qwk([1,2,3,4,5], [1,2,3,4,5]))  # should be 1.0

def evaluate(model, loader, split_name="val"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].float().to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = outputs.logits.squeeze(-1)
            all_preds.extend(preds.float().cpu().numpy())
            all_labels.extend(labels.float().cpu().numpy())
    qwk = compute_qwk(all_preds, all_labels)
    mse = np.mean((np.array(all_preds) - np.array(all_labels)) ** 2)
    print(f"  [{split_name}] QWK: {qwk:.4f} | MSE: {mse:.4f}")
    return qwk, all_preds, all_labels

QWK sanity check: 1.0


In [ ]:
print("Precomputing base similarity matrix...")
train_embs_norm  = train_embs / np.linalg.norm(train_embs, axis=1, keepdims=True)
guide_embs_norm  = guide_embs / np.linalg.norm(guide_embs, axis=1, keepdims=True)
sim_matrix_base  = train_embs_norm @ guide_embs_norm.T
n_train, n_guide = sim_matrix_base.shape
print(f"  Shape: {sim_matrix_base.shape} | Range: [{sim_matrix_base.min():.3f}, {sim_matrix_base.max():.3f}]")

def build_topk_matrix(sim_matrix, k):
    """
    Build row-normalized top-K similarity matrix.
    k='all' uses every guide point (no masking).
    """
    n_tr, n_gu = sim_matrix.shape
    topk_mat   = np.zeros_like(sim_matrix)
    k_actual   = n_gu if k == 'all' else min(k, n_gu)

    if k == 'all':
        topk_mat = np.clip(sim_matrix.copy(), 0, None)
    else:
        topk_idx = np.argpartition(sim_matrix, -k_actual, axis=1)[:, -k_actual:]
        rows     = np.arange(n_tr)[:, None]
        topk_mat[rows, topk_idx] = sim_matrix[rows, topk_idx]
        topk_mat = np.clip(topk_mat, 0, None)

    # Row-normalize → weighted average of guide losses
    row_sums = topk_mat.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    topk_mat = topk_mat / row_sums

    # Sanity — uniform guide losses should give all weights = 1.0
    dummy    = np.ones(n_gu)
    test_w   = topk_mat @ dummy
    print(f"  k={str(k):<6} k_actual={k_actual:<6} | "
          f"sanity min={test_w.min():.4f} max={test_w.max():.4f} mean={test_w.mean():.4f}")
    return topk_mat

Precomputing base similarity matrix...
  Shape: (20854, 3000) | Range: [-0.164, 0.906]


In [ ]:
all_run_results = {}

for K in K_VALUES:
    run_name = f"k_{K}"
    save_dir = f"{BASE_DIR}/{run_name}"
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  RUN: {run_name}  |  max_epochs={NUM_EPOCHS}  |  patience={PATIENCE}")
    print(f"{'='*65}")

    # Build topk matrix for this K
    topk_matrix = build_topk_matrix(sim_matrix_base, K)

    # Fresh model every run
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, ignore_mismatched_sizes=True, torch_dtype=DTYPE
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    loss_fn   = nn.MSELoss()

    # Fresh loaders every run — uniform weights to start
    sample_weights = np.ones(len(ds_tokenized["train"]))
    train_sampler  = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    train_loader   = DataLoader(ds_tokenized["train"],      batch_size=BATCH_SIZE, sampler=train_sampler)
    val_loader     = DataLoader(ds_tokenized["validation"], batch_size=BATCH_SIZE, shuffle=False)
    guide_loader   = DataLoader(ds_tokenized["guide"],      batch_size=BATCH_SIZE, shuffle=False)
    test_loader    = DataLoader(ds_tokenized["test"],       batch_size=BATCH_SIZE, shuffle=False)

    total_steps = NUM_EPOCHS * len(train_loader)
    scheduler   = get_scheduler(
        "linear", optimizer=optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    history = {
        "k": str(K), "train_loss": [], "synth_val_qwk": [],
        "real_guide_qwk": [], "real_test_qwk": [], "weight_stats": []
    }
    best_test_qwk     = -1
    best_epoch        = -1
    epochs_no_improve = 0

    for epoch in range(NUM_EPOCHS):

        # ── Reweighting (skip epoch 0 → uniform first pass) ──────────────────
        if epoch > 0:
            model.eval()
            guide_losses = []
            with torch.no_grad():
                for batch in guide_loader:
                    input_ids      = batch["input_ids"].to(DEVICE)
                    attention_mask = batch["attention_mask"].to(DEVICE)
                    labels         = batch["label"].to(DEVICE).to(DTYPE)
                    outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds          = outputs.logits.squeeze(-1).to(DTYPE)
                    guide_losses.extend(((preds - labels) ** 2).float().cpu().numpy())

            guide_losses = np.array(guide_losses)
            raw_weights  = topk_matrix @ guide_losses

            # V1b minmax normalize → [0.5, 2.0]
            w_min = raw_weights.min()
            w_max = raw_weights.max()
            raw_weights = 0.5 + 1.5 * (raw_weights - w_min) / (w_max - w_min) \
                          if w_max > w_min else np.ones_like(raw_weights)

            sample_weights = raw_weights / raw_weights.sum() * len(raw_weights)

            history["weight_stats"].append({
                "epoch":           epoch,
                "guide_loss_mean": float(guide_losses.mean()),
                "guide_loss_std":  float(guide_losses.std()),
                "w_std":           float(sample_weights.std()),
                "w_max":           float(sample_weights.max()),
            })

            # Per-star guide loss
            guide_label_arr = np.array(ds["guide"]["label"])
            star_losses = {
                int(s): float(guide_losses[guide_label_arr == s].mean())
                for s in [1,2,3,4,5]
            }
            print(f"  Guide losses — mean:{guide_losses.mean():.4f} std:{guide_losses.std():.4f} | "
                  f"per star: { {k: f'{v:.3f}' for k,v in star_losses.items()} }")
            print(f"  Weights      — std:{sample_weights.std():.4f} max:{sample_weights.max():.4f}")

            train_sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
            train_loader  = DataLoader(ds_tokenized["train"], batch_size=BATCH_SIZE, sampler=train_sampler)

        # ── Training ─────────────────────────────────────────────────────────
        model.train()
        total_loss = 0
        loop = tqdm(train_loader, desc=f"[{run_name}] Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
        for batch in loop:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].to(DEVICE).to(DTYPE)
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.squeeze(-1).to(DTYPE)
            loss    = loss_fn(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"\n  [{run_name}] Epoch {epoch+1:>2}/{NUM_EPOCHS} | loss={avg_loss:.4f}")

        synth_val_qwk,  _, _ = evaluate(model, val_loader,   "synth_val ")
        real_guide_qwk, _, _ = evaluate(model, guide_loader, "real_guide")
        real_test_qwk,  _, _ = evaluate(model, test_loader,  "real_test ")

        history["train_loss"].append(avg_loss)
        history["synth_val_qwk"].append(synth_val_qwk)
        history["real_guide_qwk"].append(real_guide_qwk)
        history["real_test_qwk"].append(real_test_qwk)

        # ── Best model + early stopping ───────────────────────────────────────
        if real_test_qwk > best_test_qwk:
            best_test_qwk     = real_test_qwk
            best_epoch        = epoch + 1
            epochs_no_improve = 0
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
            print(f"  ✓ Best saved — test QWK: {best_test_qwk:.4f} (epoch {best_epoch})")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")
            if epochs_no_improve >= PATIENCE:
                print(f"  ✗ Early stop at epoch {epoch+1}")
                break

    # Per-run save + summary
    history["best_test_qwk"] = best_test_qwk
    history["best_epoch"]    = best_epoch
    with open(f"{save_dir}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    all_run_results[str(K)] = {
        "best_qwk": best_test_qwk, "best_epoch": best_epoch,
        "delta":    best_test_qwk - BASELINE_QWK
    }

    print(f"\n  {run_name} COMPLETE | best QWK: {best_test_qwk:.4f} "
          f"| delta: {best_test_qwk - BASELINE_QWK:+.4f}")

    # Free GPU memory between runs
    del model
    torch.cuda.empty_cache()

# ── Final summary ─────────────────────────────────────────────────────────────
with open(f"{BASE_DIR}/ablation_summary.json", "w") as f:
    json.dump(all_run_results, f, indent=2)

print(f"\n{'='*65}")
print(f"  K ABLATION COMPLETE")
print(f"{'='*65}")
print(f"  Baseline (V0):  {BASELINE_QWK:.4f}")
print(f"  V1b (K=50):     0.8228")
print()
print(f"  {'K':<10} {'Best QWK':<14} {'Best Epoch':<14} {'Delta':<10}")
print(f"  {'-'*50}")
best_overall = max(all_run_results.values(), key=lambda x: x['best_qwk'])
for k, res in all_run_results.items():
    marker = " ← best" if res['best_qwk'] == best_overall['best_qwk'] else ""
    print(f"  {k:<10} {res['best_qwk']:<14.4f} {res['best_epoch']:<14} {res['delta']:+.4f}{marker}")


  RUN: k_1  |  max_epochs=10  |  patience=2
  k=1      k_actual=1      | sanity min=1.0000 max=1.0000 mean=1.0000


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias       

...

  ✓ Best saved — test QWK: 0.8184 (epoch 6)
  Guide losses — mean:0.6301 std:1.1304 | per star: {1: '0.411', 2: '0.344', 3: '0.772', 4: '0.723', 5: '0.901'}
  Weights      — std:0.1374 max:1.4746


[k_all] Epoch 7/10:   0%|          | 0/326 [00:00<?, ?it/s]


  [k_all] Epoch  7/10 | loss=0.2139
  [synth_val ] QWK: 0.9346 | MSE: 0.2520
  [real_guide] QWK: 0.8095 | MSE: 0.6402
  [real_test ] QWK: 0.8137 | MSE: 0.6213
  No improvement (1/2)
  Guide losses — mean:0.6401 std:1.1436 | per star: {1: '0.414', 2: '0.331', 3: '0.761', 4: '0.740', 5: '0.955'}
  Weights      — std:0.1374 max:1.4579


[k_all] Epoch 8/10:   0%|          | 0/326 [00:00<?, ?it/s]


  [k_all] Epoch  8/10 | loss=0.2093
  [synth_val ] QWK: 0.9345 | MSE: 0.2513
  [real_guide] QWK: 0.8112 | MSE: 0.6323
  [real_test ] QWK: 0.8145 | MSE: 0.6147
  No improvement (2/2)
  ✗ Early stop at epoch 8

  k_all COMPLETE | best QWK: 0.8184 | delta: +0.0133

  K ABLATION COMPLETE
  Baseline (V0):  0.8051
  V1b (K=50):     0.8228

  K          Best QWK       Best Epoch     Delta     
  --------------------------------------------------
  1          0.8098         5              +0.0047
  10         0.8165         7              +0.0114
  50         0.8054         8              +0.0003
  100        0.8191         6              +0.0140 ← best
  all        0.8184         6              +0.0133


In [ ]:
!zip -r /content/drive/MyDrive/beacon_yelp/k_ablation_models.zip /content/k_ablation/

  adding: content/k_ablation/ (stored 0%)
  adding: content/k_ablation/k_1/ (stored 0%)
  adding: content/k_ablation/k_1/tokenizer.json (deflated 79%)
  adding: content/k_ablation/k_1/model.safetensors (deflated 21%)
  adding: content/k_ablation/k_1/tokenizer_config.json (deflated 48%)
  adding: content/k_ablation/k_1/config.json (deflated 55%)
  adding: content/k_ablation/k_1/history.json (deflated 65%)
  adding: content/k_ablation/k_all/ (stored 0%)
  adding: content/k_ablation/k_all/tokenizer.json (deflated 79%)
  adding: content/k_ablation/k_all/model.safetensors (deflated 21%)
  adding: content/k_ablation/k_all/tokenizer_config.json (deflated 48%)
  adding: content/k_ablation/k_all/config.json (deflated 55%)
  adding: content/k_ablation/k_all/history.json (deflated 65%)
  adding: content/k_ablation/k_50/ (stored 0%)
  adding: content/k_ablation/k_50/tokenizer.json (deflated 79%)
  adding: content/k_ablation/k_50/model.safetensors (deflated 21%)
  adding: content/k_ablation/k_50/to

In [ ]:
!mv

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from datasets import Dataset, DatasetDict
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm

MODEL_NAME  = "microsoft/deberta-v3-small"
MAX_LEN     = 256
BATCH_SIZE  = 32
NUM_EPOCHS  = 5
LR          = 2e-5

# Reweighting hyperparameters
TOP_K       = 50    # number of guide neighbors per synthetic sample
TEMPERATURE = 0.1   # sharpness of weight distribution — lower = more peaked

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 8:
        DTYPE = torch.bfloat16
    else:
        DTYPE = torch.float16
else:
    DTYPE = torch.float32

print(f"Device:     {DEVICE}")
print(f"GPU:        {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Dtype:      {DTYPE}")
print(f"Top-K:      {TOP_K}")
print(f"Temp:       {TEMPERATURE}")

Device:     cuda
GPU:        NVIDIA A100-SXM4-40GB
Dtype:      torch.bfloat16
Top-K:      50
Temp:       0.1


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

ds_tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization done.")
print(ds_tokenized)

Map:   0%|          | 0/20854 [00:00<?, ? examples/s]

Map:   0%|          | 0/8937 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Tokenization done.
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20854
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8937
    })
    guide: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30000
    })
})


In [ ]:
def compute_qwk(preds, labels):
    # Clip predictions to valid range and round to nearest integer for QWK
    preds_rounded = np.clip(np.round(preds), 1, 5).astype(int)
    labels_int = np.array(labels).astype(int)
    return cohen_kappa_score(labels_int, preds_rounded, weights="quadratic")

# Quick sanity check
print("QWK sanity check:", compute_qwk([1,2,3,4,5], [1,2,3,4,5]))  # should be 1.0

QWK sanity check: 1.0


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True,
    torch_dtype=DTYPE
)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

sample_weights = np.ones(len(ds_tokenized["train"]))

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(ds_tokenized["train"],      batch_size=BATCH_SIZE, sampler=train_sampler)
val_loader   = DataLoader(ds_tokenized["validation"], batch_size=BATCH_SIZE, shuffle=False)
guide_loader = DataLoader(ds_tokenized["guide"],      batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(ds_tokenized["test"],       batch_size=BATCH_SIZE, shuffle=False)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

loss_fn = nn.MSELoss()
print(f"Model dtype:      {next(model.parameters()).dtype}")
print(f"Steps per epoch:  {len(train_loader)}")
print(f"Total steps:      {total_steps}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Model dtype:      torch.bfloat16
Steps per epoch:  652
Total steps:      3260


In [ ]:
print("Precomputing similarity matrix...")
train_embs_norm = train_embs / np.linalg.norm(train_embs, axis=1, keepdims=True)
guide_embs_norm = guide_embs / np.linalg.norm(guide_embs, axis=1, keepdims=True)

sim_matrix = train_embs_norm @ guide_embs_norm.T
print(f"  sim_matrix shape: {sim_matrix.shape}")
print(f"  sim range: [{sim_matrix.min():.3f}, {sim_matrix.max():.3f}]")

# Within-star top-K masking
# Each synthetic sample only attends to guide points with the same star rating
print(f"Applying within-star top-{TOP_K} mask...")

train_labels = np.array(ds["train"]["label"])
guide_labels = np.array(ds["guide"]["label"])

topk_matrix = np.zeros_like(sim_matrix)

for star in [1.0, 2.0, 3.0, 4.0, 5.0]:
    train_idx = np.where(train_labels == star)[0]  # synthetic samples of this star
    guide_idx = np.where(guide_labels == star)[0]  # guide samples of this star

    if len(train_idx) == 0 or len(guide_idx) == 0:
        print(f"  {int(star)}★ skipped — no samples")
        continue

    # Extract the sub-block for this star rating
    sub_sim = sim_matrix[np.ix_(train_idx, guide_idx)]  # shape: [n_train_star, n_guide_star]

    # Top-K within this star's guide points only
    k = min(TOP_K, len(guide_idx))  # can't take more neighbors than guide samples exist
    sub_topk = np.zeros_like(sub_sim)
    topk_idx = np.argpartition(sub_sim, -k, axis=1)[:, -k:]
    rows = np.arange(len(sub_sim))[:, None]
    sub_topk[rows, topk_idx] = sub_sim[rows, topk_idx]

    # Place back into full matrix
    topk_matrix[np.ix_(train_idx, guide_idx)] = sub_topk
    print(f"  {int(star)}★  train: {len(train_idx)} | guide: {len(guide_idx)} | k: {k}")

# Row-normalize
row_sums = topk_matrix.sum(axis=1, keepdims=True)
row_sums = np.where(row_sums == 0, 1, row_sums)
topk_matrix = topk_matrix / row_sums

# Sanity check — with uniform guide losses weights should all be 1.0
dummy = np.ones(guide_embs.shape[0])
test_weights = topk_matrix @ dummy
print(f"\nSanity check (uniform → all weights ≈ 1.0):")
print(f"  Min: {test_weights.min():.4f} | Max: {test_weights.max():.4f} | Mean: {test_weights.mean():.4f}")

Precomputing similarity matrix...
  sim_matrix shape: (20854, 3000)
  sim range: [-0.164, 0.906]
Applying within-star top-50 mask...
  1★  train: 4149 | guide: 600 | k: 50
  2★  train: 4220 | guide: 600 | k: 50
  3★  train: 4203 | guide: 600 | k: 50
  4★  train: 4167 | guide: 600 | k: 50
  5★  train: 4115 | guide: 600 | k: 50

Sanity check (uniform → all weights ≈ 1.0):
  Min: 1.0000 | Max: 1.0000 | Mean: 1.0000


In [ ]:
import matplotlib.pyplot as plt

# ── Raw similarity stats ──────────────────────────────────────────────
print("=== RAW SIMILARITY STATS (before temperature) ===")
nonzero_sims = sim_matrix[sim_matrix > 0]
print(f"Min:    {nonzero_sims.min():.4f}")
print(f"Max:    {nonzero_sims.max():.4f}")
print(f"Mean:   {nonzero_sims.mean():.4f}")
print(f"Std:    {nonzero_sims.std():.4f}")
print(f"Percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>3}: {np.percentile(nonzero_sims, p):.4f}")

# ── Weight distribution after temp scaling ────────────────────────────
# Compute raw weights using uniform guide losses = 1.0 (just to inspect shape)
dummy_guide_losses = np.ones(guide_embs.shape[0])
raw_weights = topk_matrix @ dummy_guide_losses

print("\n=== WEIGHT DISTRIBUTION (uniform guide losses) ===")
print(f"Min:    {raw_weights.min():.4f}")
print(f"Max:    {raw_weights.max():.4f}")
print(f"Mean:   {raw_weights.mean():.4f}")
print(f"Std:    {raw_weights.std():.4f}")
print(f"Ratio max/min: {raw_weights.max()/raw_weights.min():.1f}x")
print(f"Percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>3}: {np.percentile(raw_weights, p):.4f}")

# How many samples get >10x the mean weight?
mean_w = raw_weights.mean()
dominated = (raw_weights > 10 * mean_w).sum()
print(f"\nSamples with weight >10x mean: {dominated} ({100*dominated/len(raw_weights):.1f}%)")
print(f"Samples with weight <0.1x mean: {(raw_weights < 0.1 * mean_w).sum()}")

# ── Plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Raw similarity distribution
axes[0].hist(nonzero_sims, bins=50, color='steelblue', edgecolor='none')
axes[0].set_title("Raw Cosine Similarities\n(top-K nonzero entries)")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Count")

# 2. Weight distribution (log scale to see the tail)
axes[1].hist(raw_weights, bins=100, color='coral', edgecolor='none')
axes[1].set_title("Raw Weights (uniform guide losses)\nlinear scale")
axes[1].set_xlabel("Weight")
axes[1].set_ylabel("Count")

# 3. Same but log scale
axes[2].hist(raw_weights, bins=100, color='coral', edgecolor='none', log=True)
axes[2].set_title("Raw Weights\nlog y-scale")
axes[2].set_xlabel("Weight")
axes[2].set_ylabel("Count (log)")

plt.tight_layout()
plt.savefig("/content/weight_distribution.png", dpi=150)
plt.show()
print("Saved to /content/weight_distribution.png")

In [ ]:
def evaluate(loader, split_name="val"):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].float().to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.squeeze(-1)

            # cast to float32 before moving to cpu — fixes bf16/fp16 numpy error
            all_preds.extend(preds.float().cpu().numpy())
            all_labels.extend(labels.float().cpu().numpy())

    qwk = compute_qwk(all_preds, all_labels)
    mse = np.mean((np.array(all_preds) - np.array(all_labels)) ** 2)
    print(f"  [{split_name}] QWK: {qwk:.4f} | MSE: {mse:.4f}")
    return qwk, all_preds, all_labels

In [ ]:
import os, json

SAVE_DIR = "/content/reweighted_model_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

history = {
    "train_loss":     [],
    "synth_val_qwk":  [],
    "real_guide_qwk": [],
    "real_test_qwk":  [],
    "weight_stats":   []
}

best_test_qwk = -1
best_epoch    = -1

for epoch in range(NUM_EPOCHS):

    # ── REWEIGHTING STEP ──────────────────────────────────────────────────────
    if epoch > 0:
        print(f"\n  Computing guide losses for reweighting...")
        model.eval()
        guide_losses = []

        with torch.no_grad():
            for batch in guide_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels         = batch["label"].to(DEVICE).to(DTYPE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds   = outputs.logits.squeeze(-1).to(DTYPE)

                # Per-sample loss — shape: [batch_size]
                per_sample_loss = (preds - labels) ** 2
                guide_losses.extend(per_sample_loss.float().cpu().numpy())

        guide_losses = np.array(guide_losses)  # shape: [n_guide]

        # Transfer guide signal to synthetic samples via row-normalized sim matrix
        raw_weights = topk_matrix @ guide_losses  # shape: [n_train]

        # NEW — replace with this
        w_min = raw_weights.min()
        w_max = raw_weights.max()
        raw_weights = 0.5 + 1.5 * (raw_weights - w_min) / (w_max - w_min)

        # Normalize so weights sum to n_train — expected samples per epoch unchanged
        sample_weights = raw_weights / raw_weights.sum() * len(raw_weights)

        # Log weight stats
        stats = {
            "epoch":           epoch,
            "w_mean":          float(sample_weights.mean()),
            "w_std":           float(sample_weights.std()),
            "w_min":           float(sample_weights.min()),
            "w_max":           float(sample_weights.max()),
            "guide_loss_mean": float(guide_losses.mean()),
            "guide_loss_std":  float(guide_losses.std()),
            "guide_loss_min":  float(guide_losses.min()),
            "guide_loss_max":  float(guide_losses.max()),
        }
        history["weight_stats"].append(stats)

        print(f"  Guide losses  — mean: {stats['guide_loss_mean']:.4f} | "
              f"std: {stats['guide_loss_std']:.4f} | "
              f"min: {stats['guide_loss_min']:.4f} | "
              f"max: {stats['guide_loss_max']:.4f}")
        print(f"  Sample weights — mean: {stats['w_mean']:.4f} | "
              f"std: {stats['w_std']:.4f} | "
              f"min: {stats['w_min']:.4f} | "
              f"max: {stats['w_max']:.4f}")

        # Star-level breakdown — see which star ratings are driving high losses
        guide_labels = np.array(ds["guide"]["label"])
        print(f"  Per-star guide loss:")
        for star in [1.0, 2.0, 3.0, 4.0, 5.0]:
            mask = guide_labels == star
            if mask.sum() > 0:
                print(f"    {int(star)}★  mean loss: {guide_losses[mask].mean():.4f} "
                      f"| n: {mask.sum()}")

        # Rebuild sampler and loader with new weights
        train_sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        train_loader = DataLoader(
            ds_tokenized["train"],
            batch_size=BATCH_SIZE,
            sampler=train_sampler
        )
    # ─────────────────────────────────────────────────────────────────────────

    # ── TRAINING STEP ─────────────────────────────────────────────────────────
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in loop:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE).to(DTYPE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.squeeze(-1).to(DTYPE)

        loss = loss_fn(preds, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")
    # ─────────────────────────────────────────────────────────────────────────

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | Avg Train Loss: {avg_loss:.4f}")

    synth_val_qwk,  _, _ = evaluate(val_loader,   "synth_val ")
    real_guide_qwk, _, _ = evaluate(guide_loader, "real_guide")
    real_test_qwk,  _, _ = evaluate(test_loader,  "real_test ")

    history["train_loss"].append(avg_loss)
    history["synth_val_qwk"].append(synth_val_qwk)
    history["real_guide_qwk"].append(real_guide_qwk)
    history["real_test_qwk"].append(real_test_qwk)

    if real_test_qwk > best_test_qwk:
        best_test_qwk = real_test_qwk
        best_epoch    = epoch + 1
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✓ Best model saved (test QWK: {best_test_qwk:.4f})")

# ── SAVE HISTORY ──────────────────────────────────────────────────────────────
with open(f"{SAVE_DIR}/history.json", "w") as f:
    json.dump(history, f, indent=2)

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f"\n=== V2 REWEIGHTING COMPLETE ===")
print(f"Best real_test QWK: {best_test_qwk:.4f} at epoch {best_epoch}")

print(f"\n{'Epoch':<8}{'Train Loss':<14}{'Synth Val':<14}{'Real Guide':<14}{'Real Test':<14}")
print("-" * 60)
for i in range(NUM_EPOCHS):
    marker = " ←" if (i + 1) == best_epoch else ""
    print(f"{i+1:<8}{history['train_loss'][i]:<14.4f}{history['synth_val_qwk'][i]:<14.4f}"
          f"{history['real_guide_qwk'][i]:<14.4f}{history['real_test_qwk'][i]:<14.4f}{marker}")

print(f"\nBaseline best: 0.8051 | V1 best: 0.8082 | V1b best: 0.8228 | V2 best: {best_test_qwk:.4f} | "
      f"Delta: {best_test_qwk - 0.8051:+.4f}")

# ── WEIGHT EVOLUTION SUMMARY ──────────────────────────────────────────────────
if history["weight_stats"]:
    print(f"\nWeight evolution across epochs:")
    print(f"{'Epoch':<8}{'Guide Loss Mean':<18}{'W Mean':<12}{'W Std':<12}{'W Max':<12}")
    print("-" * 60)
    for s in history["weight_stats"]:
        print(f"{s['epoch']:<8}{s['guide_loss_mean']:<18.4f}"
              f"{s['w_mean']:<12.4f}{s['w_std']:<12.4f}{s['w_max']:<12.4f}")

Epoch 1/5:   0%|          | 0/652 [00:00<?, ?it/s]


Epoch 1/5 | Avg Train Loss: 3.5717
  [synth_val ] QWK: 0.8874 | MSE: 0.3908
  [real_guide] QWK: 0.7933 | MSE: 0.6982
  [real_test ] QWK: 0.7942 | MSE: 0.6901


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (test QWK: 0.7942)

  Computing guide losses for reweighting...
  Guide losses  — mean: 0.6982 | std: 1.2164 | min: 0.0000 | max: 12.6875
  Sample weights — mean: 1.0000 | std: 0.1955 | min: 0.6408 | max: 2.5634
  Per-star guide loss:
    1★  mean loss: 0.4723 | n: 600
    2★  mean loss: 0.3940 | n: 600
    3★  mean loss: 0.8534 | n: 600
    4★  mean loss: 0.7618 | n: 600
    5★  mean loss: 1.0091 | n: 600


Epoch 2/5:   0%|          | 0/652 [00:00<?, ?it/s]

...


Epoch 4/5 | Avg Train Loss: 0.2369
  [synth_val ] QWK: 0.9243 | MSE: 0.2840
  [real_guide] QWK: 0.8123 | MSE: 0.6432
  [real_test ] QWK: 0.8127 | MSE: 0.6350


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (test QWK: 0.8127)

  Computing guide losses for reweighting...
  Guide losses  — mean: 0.6431 | std: 1.1470 | min: 0.0000 | max: 13.8125
  Sample weights — mean: 1.0000 | std: 0.1986 | min: 0.6215 | max: 2.4861
  Per-star guide loss:
    1★  mean loss: 0.4926 | n: 600
    2★  mean loss: 0.3737 | n: 600
    3★  mean loss: 0.7900 | n: 600
    4★  mean loss: 0.7417 | n: 600
    5★  mean loss: 0.8176 | n: 600


Epoch 5/5:   0%|          | 0/652 [00:00<?, ?it/s]


Epoch 5/5 | Avg Train Loss: 0.2386
  [synth_val ] QWK: 0.9263 | MSE: 0.2792
  [real_guide] QWK: 0.8106 | MSE: 0.6545
  [real_test ] QWK: 0.8123 | MSE: 0.6450

=== V2 REWEIGHTING COMPLETE ===
Best real_test QWK: 0.8127 at epoch 4

Epoch   Train Loss    Synth Val     Real Guide    Real Test     
------------------------------------------------------------
1       3.5717        0.8874        0.7933        0.7942        
2       0.2846        0.9246        0.8020        0.8033        
3       0.2412        0.9253        0.8104        0.8121        
4       0.2369        0.9243        0.8123        0.8127         ←
5       0.2386        0.9263        0.8106        0.8123        

Baseline best: 0.8051 | V1 best: 0.8082 | V1b best: 0.8228 | V2 best: 0.8127 | Delta: +0.0076

Weight evolution across epochs:
Epoch   Guide Loss Mean   W Mean      W Std       W Max       
------------------------------------------------------------
1       0.6982            1.0000      0.1955      2.5634      
2

In [ ]:
!zip /content/baseline_model.zip -r /content/baseline_model

In [ ]:
!ls


baseline_model	    restaurant_data_package.zip  reweighted_model_v2
baseline_model.zip  reweighted_model_v1		 sample_data
dataset		    reweighted_model_v1b	 weight_distribution.png


In [ ]:
!zip -r models_archive.zip baseline_model reweighted_model*

  adding: baseline_model/ (stored 0%)
  adding: baseline_model/model.safetensors (deflated 21%)
  adding: baseline_model/config.json (deflated 55%)
  adding: baseline_model/tokenizer_config.json (deflated 48%)
  adding: baseline_model/tokenizer.json (deflated 79%)
  adding: baseline_model/history.json (deflated 52%)
  adding: reweighted_model_v1/ (stored 0%)
  adding: reweighted_model_v1/model.safetensors (deflated 21%)
  adding: reweighted_model_v1/config.json (deflated 55%)
  adding: reweighted_model_v1/tokenizer_config.json (deflated 48%)
  adding: reweighted_model_v1/tokenizer.json (deflated 79%)
  adding: reweighted_model_v1/history.json (deflated 66%)
  adding: reweighted_model_v1b/ (stored 0%)
  adding: reweighted_model_v1b/model.safetensors (deflated 21%)
  adding: reweighted_model_v1b/config.json (deflated 55%)
  adding: reweighted_model_v1b/tokenizer_config.json (deflated 48%)
  adding: reweighted_model_v1b/tokenizer.json (deflated 79%)
  adding: reweighted_model_v1b/history.